# Домашнее задание к семинару 14 (HW14)

Тема: эмбеддинги, индекс `FAISS`, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

HW14 выполняется в личном репозитории студента (на основе шаблона курса) в папке `homeworks/HW14/`.

---

## 1. Цель

Закрепить:

- построение векторных представлений текстовых фрагментов;
- работу с индексом `FAISS` для поиска по базе знаний;
- базовую оценку качества извлечения (`retrieval`);
- обновление базы знаний и переиндексацию;
- сборку простого mini-RAG-контура;
- аккуратное оформление результата: один ноутбук, короткий отчёт, минимальные артефакты.

---

## 2. Задание

### 2.1. Структура для HW14 (обязательно)

1. В корне репозитория должна быть папка `homeworks/` (создать, если её ещё нет).
2. Внутри `homeworks/` создать папку `HW14/`.
3. В папке `homeworks/HW14/` создать:

- основной ноутбук: `HW14.ipynb`
- отчёт: `report.md`
- папку для артефактов: `artifacts/`

> Имена папок и файлов должны быть строго такими, как указано (регистр важен).

---

### 2.2. База знаний для домашней работы

Нужно выбрать **одну** небольшую текстовую базу знаний.

Подходящие варианты:

- подборка коротких тематических статей;
- FAQ или справочные материалы;
- набор внутренних регламентов / инструкций / заметок;
- markdown / txt / csv / json с текстовыми описаниями;
- любой другой **готовый** набор документов, если он подходит для retrieval-задачи.

Рекомендуемый масштаб:

- **10-30 документов**;
- или такой объём, который после чанкинга даёт примерно **30-150 текстовых фрагментов**.

Требования к базе знаний:

- документы должны относиться к одной понятной теме;
- документы должны быть достаточно содержательными, чтобы по ним имело смысл задавать вопросы;
- база знаний не должна быть слишком большой для обычного учебного запуска;
- база знаний должна быть загружена и обработана воспроизводимо.

Важно:

- **не требуется** искать большой промышленный датасет;
- **не требуется** поднимать внешнюю векторную БД;
- главное – корректно пройти учебный pipeline: чанки → эмбеддинги → индекс → retrieval → оценка → обновление → mini-RAG.

---

### 2.3. Содержание ноутбука `HW14.ipynb` (обязательно)

В ноутбуке `homeworks/HW14/HW14.ipynb` реализуйте и покажите следующие блоки.

#### 2.3.1. Импорты, seed и среда

Нужно:

1. Импортировать библиотеки: `numpy`, `pandas`, `sklearn`, `matplotlib`, `faiss` (или `faiss-cpu`) и всё, что нужно по делу.
2. Зафиксировать seed (минимум `random`, `numpy`; если используете `torch`, то и `torch`).
3. Если используется модель эмбеддингов на `torch`, определить устройство (`cuda` при наличии, иначе `cpu`) и использовать его последовательно.

In [1]:
# 2.3.1 Imports, seed and environment

import importlib
import subprocess
import sys, os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from typing import List, Dict, Tuple, Optional
from IPython.display import display, Markdown

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")

try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("Не удалось импортировать faiss:", repr(e))

# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("NumPy:", np.__version__) 
print("Pandas:", pd.__version__) 
print("FAISS available:", FAISS_AVAILABLE) 
print("Устройство для работы:", DEVICE)

NumPy: 2.4.2
Pandas: 3.0.1
FAISS available: True
Устройство для работы: cpu


#### 2.3.2. База знаний и первичный анализ

Нужно:

- загрузить выбранную базу знаний;
- показать число документов;
- вывести 3-5 примеров документов или их фрагментов;
- кратко пояснить, что это за предметная область и почему по ней разумно строить retrieval / mini-RAG.

In [2]:
# 2.3.2. База знаний и первичный анализ

documents = [
    {
        "doc_id": "cat_001",
        "title": "Происхождение домашней кошки",
        "text": """Домашняя кошка относится к виду Felis catus. Считается, что предком домашней кошки была
ближневосточная дикая кошка Felis silvestris lybica. Одомашнивание кошек происходило постепенно,
когда животные начали жить рядом с человеческими поселениями, где было много зерна и грызунов.
Люди не приручали кошек так же активно, как собак, поэтому домашние кошки сохранили много черт
самостоятельного хищника. Генетические исследования показывают, что современная домашняя кошка
сохраняет близкое родство с дикими предками. Именно поэтому кошки даже в домашних условиях
демонстрируют охотничье поведение, территориальность и выраженный интерес к движущимся объектам.
Изучение происхождения кошек важно для понимания их поведения и адаптации к жизни рядом с человеком."""
    },
    {
        "doc_id": "cat_002",
        "title": "Особенности скелета и гибкости кошки",
        "text": """Кошки известны своей гибкостью и способностью быстро менять положение тела. Это связано
с особенностями строения их позвоночника, суставов и плечевого пояса. У кошки очень подвижный
позвоночник, который позволяет ей сгибаться и разворачиваться во время прыжков и падений.
Ключицы у кошек развиты иначе, чем у человека, поэтому передние конечности двигаются свободнее.
Благодаря этому кошка может пролезать в узкие пространства, если туда проходит её голова.
Гибкость тела помогает кошке во время охоты, лазания и прыжков. Однако гибкость не означает
полную неуязвимость: при неудачных падениях и травмах кошки тоже могут получать повреждения.
С точки зрения биомеханики кошка является очень эффективным мелким хищником."""
    },
    {
        "doc_id": "cat_003",
        "title": "Почему кошки хорошо видят в сумерках",
        "text": """Зрение кошек приспособлено к активности при слабом освещении. В сетчатке кошки много
палочек — клеток, которые хорошо работают в сумерках. Кроме того, у кошек есть отражающий слой
tapetum lucidum, который усиливает использование света и помогает видеть при низкой освещённости.
Именно из-за этого глаза кошек могут ярко светиться в темноте при попадании света. При этом
кошки различают цвета хуже человека и видят мир менее насыщенным по оттенкам. Их зрение лучше
подходит для обнаружения движения, чем для распознавания мелких неподвижных деталей. Такая
особенность полезна для охоты на мелкую добычу. С точки зрения эволюции это связано с образом
жизни сумеречного хищника, который наиболее активен утром и вечером."""
    },
    {
        "doc_id": "cat_004",
        "title": "Слух кошек и ориентация на звук",
        "text": """Слух у кошек очень чувствительный и играет важную роль в охоте и ориентировании в пространстве.
Кошки способны улавливать высокочастотные звуки, в том числе писк мелких грызунов. Их ушные раковины
могут поворачиваться независимо, что помогает точнее определять направление источника звука.
Благодаря этому кошка может быстро понять, откуда идёт шум, даже если объект скрыт от глаз.
Слух особенно важен в условиях слабого освещения, когда зрение работает не идеально. Комбинация
чувствительного слуха и хорошего зрения в сумерках делает кошку эффективным охотником. Исследования
поведения показывают, что кошки быстро реагируют на резкие и необычные звуки, но могут игнорировать
знакомые и безопасные бытовые шумы."""
    },
    {
        "doc_id": "cat_005",
        "title": "Вибриссы и пространственная ориентация",
        "text": """Вибриссы, которые часто называют усами, являются важным сенсорным инструментом кошки.
Они расположены не только на морде, но и над глазами, на подбородке и на передних лапах.
Вибриссы помогают кошке оценивать расстояние до объектов, чувствовать движение воздуха и
лучше ориентироваться в темноте. Они особенно полезны в узких пространствах и при исследовании
окружения. Вибриссы нельзя рассматривать как обычную шерсть, потому что они связаны с чувствительными
нервными структурами. Повреждение или сильное укорочение вибрисс может ухудшить ориентацию животного.
С научной точки зрения вибриссы — это важная часть сенсорной системы, которая дополняет зрение,
слух и осязание."""
    },
    {
        "doc_id": "cat_006",
        "title": "Почему кошки мурлыкают",
        "text": """Мурлыканье — одно из самых известных проявлений поведения кошек. Оно связано с ритмической
работой мышц гортани и дыхательной системы. Кошки могут мурлыкать в разных ситуациях: при комфорте,
при общении с человеком, во время кормления котят и иногда даже в стрессовых условиях. Поэтому
мурлыканье не всегда означает только удовольствие. Исследователи предполагают, что мурлыканье может
играть роль в коммуникации и самоуспокоении. Некоторые работы также обсуждают возможную связь между
вибрациями при мурлыканье и восстановительными процессами тканей, но этот вопрос изучен не полностью.
С поведенческой точки зрения мурлыканье важно анализировать в контексте всей ситуации, а не как
однозначный признак хорошего настроения."""
    },
    {
        "doc_id": "cat_007",
        "title": "Сон кошек и распределение активности",
        "text": """Кошки спят значительную часть суток, но это не значит, что они ленивы в биологическом смысле.
Многие кошки распределяют активность неравномерно и наиболее оживлены утром и вечером. Такой режим
часто называют сумеречной активностью. Большое количество сна помогает экономить энергию между
периодами активности и охоты. При этом часть времени кошка находится не в глубоком сне, а в лёгкой
дремоте, когда она сохраняет готовность быстро отреагировать на звук или движение. Продолжительность
сна может зависеть от возраста, уровня активности, окружающей среды и состояния животного. Наблюдение
за режимом сна полезно для понимания нормального поведения кошки в домашних условиях."""
    },
    {
        "doc_id": "cat_008",
        "title": "Территориальность и маркировка у кошек",
        "text": """Кошки являются территориальными животными. Они используют пространство не хаотично, а делят
его на значимые зоны: место отдыха, путь передвижения, точки наблюдения и участки с ресурсами.
Для обозначения территории кошки могут применять запаховые метки, трение мордой о предметы,
царапанье поверхностей и другие формы маркировки. Такое поведение связано не только с агрессией,
но и с созданием понятной и предсказуемой среды. В многокошачьих домах территориальные конфликты
могут усиливаться, если животным не хватает пространства и ресурсов. Изучение территориальности
важно для понимания стресса, поведения и организации среды для домашних кошек."""
    },
    {
        "doc_id": "cat_009",
        "title": "Питание кошек как облигатных хищников",
        "text": """Кошки относятся к облигатным хищникам. Это означает, что их организм эволюционно приспособлен
к рациону с высоким содержанием животного белка. У кошек есть особые потребности в некоторых веществах,
которые в природе они получают из тканей добычи. Например, для них особенно важны определённые
аминокислоты и соединения, связанные с животным происхождением пищи. Пищеварительная система кошки
лучше приспособлена к перевариванию мясных продуктов, чем к переработке большого количества
растительной пищи. С точки зрения биологии это объясняет, почему питание кошек отличается от питания
всеядных животных. Пищевое поведение домашних кошек также зависит от привычек, среды и доступности пищи."""
    },
    {
        "doc_id": "cat_010",
        "title": "Коммуникация кошек с человеком",
        "text": """Кошки общаются с человеком с помощью голоса, позы тела, движения хвоста, положения ушей
и зрительного контакта. Интересно, что мяуканье особенно активно используется именно при взаимодействии
с людьми, тогда как между взрослыми кошками оно встречается реже. Это говорит о том, что домашние
кошки адаптировали часть сигналов под общение с человеком. Медленное моргание часто рассматривают
как дружественный сигнал. Положение хвоста и ушей помогает оценить эмоциональное состояние кошки:
интерес, настороженность, возбуждение или страх. Для правильной интерпретации поведения важно смотреть
не на один сигнал, а на их сочетание. Научное изучение коммуникации кошек помогает лучше понимать
их поведение в домашних условиях."""
    },
    {
        "doc_id": "cat_011",
        "title": "Почему кошки приносят добычу или игрушки",
        "text": """Поведение, при котором кошка приносит добычу или игрушку, связано с охотничьими инстинктами
и индивидуальными особенностями поведения. У некоторых кошек это может напоминать демонстрацию добычи,
у других — приглашение к взаимодействию. Такое поведение чаще наблюдается у животных с выраженной
игровой активностью и интересом к движущимся объектам. Домашняя среда не отменяет природных
поведенческих программ, поэтому даже кошки, которые никогда не охотились на реальную добычу,
могут активно преследовать, ловить и переносить игрушки. Для этологии это интересный пример того,
как врождённые модели поведения проявляются в изменённой среде."""
    },
    {
        "doc_id": "cat_012",
        "title": "Домашняя среда и благополучие кошки",
        "text": """Благополучие домашней кошки зависит не только от питания, но и от качества среды.
Для кошек важны укрытия, вертикальные поверхности, безопасные места отдыха, возможность наблюдать
за пространством и доступ к ресурсам без постоянной конкуренции. Обогащение среды помогает снизить
стресс и уменьшить поведенческие проблемы. Например, когтеточки, полки, игровые объекты и отдельные
места кормления делают среду более подходящей для потребностей кошки. С научной точки зрения
благополучие связано с возможностью реализовывать естественные формы поведения: исследование,
отдых, наблюдение, игра и контроль над дистанцией до других животных и людей."""
    }
]

documents_df = pd.DataFrame(documents)

print("Число документов в базе знаний:", len(documents_df))
display(documents_df.head())

Число документов в базе знаний: 12


,doc_id,title,text
0,cat_001,Происхождение домашней кошки,Домашняя кошка относится к виду Felis catus. С...
1,cat_002,Особенности скелета и гибкости кошки,Кошки известны своей гибкостью и способностью ...
2,cat_003,Почему кошки хорошо видят в сумерках,Зрение кошек приспособлено к активности при сл...
3,cat_004,Слух кошек и ориентация на звук,Слух у кошек очень чувствительный и играет важ...
4,cat_005,Вибриссы и пространственная ориентация,"Вибриссы, которые часто называют усами, являют..."


In [3]:
num_examples = min(5, len(documents_df))

for i in range(num_examples):
    row = documents_df.iloc[i]
    print("=" * 100)
    print(f"Пример {i + 1}")
    print("doc_id:", row["doc_id"])
    print("title:", row["title"])
    print("fragment:", row["text"][:300], "...")

Пример 1
doc_id: cat_001
title: Происхождение домашней кошки
fragment: Домашняя кошка относится к виду Felis catus. Считается, что предком домашней кошки была
ближневосточная дикая кошка Felis silvestris lybica. Одомашнивание кошек происходило постепенно,
когда животные начали жить рядом с человеческими поселениями, где было много зерна и грызунов.
Люди не приручали ко ...
Пример 2
doc_id: cat_002
title: Особенности скелета и гибкости кошки
fragment: Кошки известны своей гибкостью и способностью быстро менять положение тела. Это связано
с особенностями строения их позвоночника, суставов и плечевого пояса. У кошки очень подвижный
позвоночник, который позволяет ей сгибаться и разворачиваться во время прыжков и падений.
Ключицы у кошек развиты инач ...
Пример 3
doc_id: cat_003
title: Почему кошки хорошо видят в сумерках
fragment: Зрение кошек приспособлено к активности при слабом освещении. В сетчатке кошки много
палочек — клеток, которые хорошо работают в сумерках. Кроме того, у кошек ес

In [4]:
documents_df["text_length_chars"] = documents_df["text"].apply(len)
documents_df["text_length_words"] = documents_df["text"].apply(lambda x: len(x.split()))

display(documents_df[["doc_id", "title", "text_length_chars", "text_length_words"]].head())

print("Статистика по длине документов:")
display(documents_df[["text_length_chars", "text_length_words"]].describe())

,doc_id,title,text_length_chars,text_length_words
0,cat_001,Происхождение домашней кошки,758,99
1,cat_002,Особенности скелета и гибкости кошки,719,99
2,cat_003,Почему кошки хорошо видят в сумерках,717,104
3,cat_004,Слух кошек и ориентация на звук,713,98
4,cat_005,Вибриссы и пространственная ориентация,675,91


Статистика по длине документов:


,text_length_chars,text_length_words
count,12.000000,12.000000
mean,693.250000,93.250000
std,37.854086,6.837397
min,639.000000,83.000000
25%,668.000000,89.500000
50%,701.000000,94.500000
75%,719.250000,98.250000
max,758.000000,104.000000


В данной домашней работе используется небольшая текстовая база знаний по теме биологии и поведения домашних кошек.  
Документы описывают происхождение кошек, особенности их зрения и слуха, роль вибрисс, причины мурлыканья, территориальность, питание, коммуникацию и влияние среды на благополучие.

Эта предметная область подходит для retrieval / mini-RAG, потому что:
1. документы относятся к одной понятной теме;
2. по ним можно задавать содержательные вопросы в свободной форме;
3. ответы удобно строить через поиск релевантных документов или фрагментов, а затем формировать итоговый ответ по найденным источникам.

Таким образом, данная база знаний подходит для учебного pipeline:
чанки → эмбеддинги → индекс → retrieval → оценка → обновление → mini-RAG.

#### 2.3.3. Чанкинг документов

Нужно:

- реализовать разбиение документов на текстовые фрагменты;
- показать, как выбранный документ превращается в чанки;
- кратко пояснить выбранные параметры (`chunk_size`, `overlap` или их аналог).

Важно:

- не требуется делать сложный продвинутый чанкинг;
- достаточно одного разумного и воспроизводимого варианта.

#### 2.3.4. Эмбеддинги и индекс `FAISS`

Нужно:

- выбрать одну embedding-модель или один корректный способ построения векторных представлений;
- получить векторы для чанков;
- построить индекс `FAISS`;
- показать поиск top-k фрагментов хотя бы для 3-5 примерных запросов.

Важно:

- в обязательной части нужен именно **векторный retrieval**;
- не требуется сравнивать несколько embedding-моделей.

#### 2.3.5. Контрольные запросы и оценка retrieval

Нужно подготовить небольшой набор контрольных запросов.

Рекомендуется:

- **8-12 запросов**;
- для каждого запроса указать ожидаемый релевантный документ, источник или фрагмент.

Нужно:

- выполнить retrieval для контрольных запросов;
- оценить качество извлечения;
- посчитать минимум:
  - `hit@k`
  - `recall@k`

Дополнительно приветствуется, но не обязательно:

- `MRR@k`;
- отдельный анализ запросов, на которых retrieval ошибается.

#### 2.3.6. Небольшой эксперимент с параметрами retrieval

Нужно выполнить **один** внятный сравнительный эксперимент.

Допустимые варианты:

- сравнить два значения `chunk_size`;
- сравнить два варианта `overlap`;
- сравнить два значения `top_k`.

Важно:

- **не требуется** большой benchmark;
- достаточно одного аккуратного сравнения с кратким выводом.

#### 2.3.7. Обновление базы знаний и переиндексация

Нужно:

- добавить в базу знаний **2-5 новых документов** или обновить часть существующих;
- повторно выполнить чанкинг / индексацию;
- показать, как retrieval меняется до и после обновления базы знаний.

Нужно привести хотя бы несколько примеров запросов, для которых видно, что обновление базы знаний действительно влияет на результат поиска.

#### 2.3.8. Mini-RAG

Нужно собрать простой mini-RAG-конвейер:

- получить запрос пользователя;
- извлечь top-k релевантных фрагментов;
- собрать контекст из найденных фрагментов;
- сформировать ответ по найденному контексту;
- вернуть ответ вместе с источниками.

Важно:

- **не требуется** строить production-grade RAG;
- **не требуется** внешний сервис или веб-интерфейс;
- **не требуется** использовать тяжёлую LLM-инфраструктуру;
- достаточно учебного, но понятного и воспроизводимого mini-RAG.

#### 2.3.9. Краткий анализ ошибок

Нужно:

- показать 3-5 примеров работы mini-RAG;
- кратко прокомментировать 2-4 неудачных или пограничных случая;
- пояснить, что именно пошло не так: retrieval, состав контекста, формулировка вопроса, неполнота базы знаний и т.д.

---

## 3. Эксперименты

### 3.1. Обязательная часть

Нужно выполнить **один** основной учебный pipeline:

- фиксированная база знаний;
- фиксированный способ чанкинга;
- фиксированная embedding-модель;
- один retrieval-конвейер с `FAISS`;
- один набор контрольных запросов;
- одна оценка retrieval;
- одно обновление базы знаний;
- один mini-RAG-конвейер.

Главное:

- воспроизводимость;
- корректный retrieval;
- понятная оценка качества;
- демонстрация влияния обновления базы знаний.

### 3.2. Что не требуется

В обязательной части **не требуется**:

- `pgvector`;
- внешняя векторная БД;
- веб-приложение;
- сравнение 3-4 embedding-моделей;
- большой подбор гиперпараметров;
- LangChain или другие тяжёлые фреймворки;
- сложный reranking;
- production-grade реализация RAG.

---

## 4. Артефакты (обязательно)

В папке `homeworks/HW14/artifacts/` должны быть:

1. `retrieval_eval.csv` – таблица с результатами retrieval на контрольных запросах.

   Минимальные поля:

   - `query`
   - `expected_source`
   - `retrieved_sources`
   - `hit_at_k`

   Дополнительно приветствуется поле:

   - `rank_of_first_relevant`

2. `rag_examples.csv` – таблица с примерами работы mini-RAG.

   Минимальные поля:

   - `question`
   - `answer`
   - `retrieved_sources`

3. `retrieval_before_after_update.csv` – сравнение retrieval до и после обновления базы знаний.

   Минимальные поля:

   - `query`
   - `before_retrieved_sources`
   - `after_retrieved_sources`
   - `changed`

Этого достаточно.

Дополнительно приветствуется, но не обязательно:

- `chunk_examples.csv`
- `retrieval_metrics_summary.json`
- `retrieval_quality_plot.png`

---

## 5. Отчёт `report.md` (обязательно)

1. В материалах семинара будет шаблон: `S14-hw-report-template.md`.
2. Нужно создать файл `homeworks/HW14/report.md` и заполнить его по шаблону.

Важно:

- не менять названия разделов (заголовков) в отчёте;
- вставлять результаты и выводы в соответствующие секции;
- в отчёте должны быть ссылки на файлы из `artifacts/`.

---

## 6. Требования к структуре и именованию (итог)

К дедлайну в репозитории должно быть:

- `homeworks/HW14/HW14.ipynb`
- `homeworks/HW14/report.md`
- `homeworks/HW14/artifacts/`
  - `retrieval_eval.csv`
  - `rag_examples.csv`
  - `retrieval_before_after_update.csv`

Требования:

- названия папок и файлов – строго как указано;
- ноутбук выполняется без ошибок при последовательном запуске всех ячеек;
- в ноутбуке есть база знаний, чанкинг, эмбеддинги, индекс `FAISS`, оценка retrieval, обновление базы знаний и mini-RAG;
- отчёт заполнен по шаблону.

---

## 7. Критерии зачёта

HW14 считается зачтённым, если:

1. Соблюдена структура `homeworks/HW14/` и нейминг файлов.
2. В `HW14.ipynb` есть:

   - загрузка выбранной базы знаний;
   - краткий sanity-check документов;
   - чанкинг и показ примеров чанков;
   - построение векторных представлений;
   - индекс `FAISS` и поиск по нескольким запросам;
   - набор контрольных запросов для retrieval;
   - расчёт минимум `hit@k` и `recall@k`;
   - один небольшой сравнительный эксперимент по параметрам retrieval;
   - обновление базы знаний и переиндексация;
   - mini-RAG с ответами по найденному контексту;
   - возврат источников вместе с ответом;
   - краткий анализ нескольких ошибок или ограничений.

3. В `artifacts/` лежат требуемые файлы: `retrieval_eval.csv`, `rag_examples.csv`, `retrieval_before_after_update.csv`.
4. Заполнен `report.md` по шаблону.

---

## 8. Опциональная часть (для желающих)

Не обязательна для зачёта, но приветствуется:

- сравнение двух embedding-моделей;
- дополнительная retrieval-метрика (`MRR@k` и др.);
- reranking;
- более подробный анализ ошибок на 8-12 примерах;
- сравнение двух способов сборки контекста для mini-RAG;
- использование компактной генеративной модели вместо простого шаблонного генератора ответа.

---

## 9. Сроки и порядок сдачи

- Работа выполняется индивидуально.
- Дедлайн объявляется преподавателем отдельно.
- Факт сдачи: к дедлайну в репозитории есть `homeworks/HW14/` со всеми файлами и корректно выполненным ноутбуком.